In [60]:
import pandas as pd
import numpy as np

housing_df = pd.read_csv('numeric_prop_census_isd.csv')


#changed year built to years since
def years_since(x):
  return 2026-x
yr_built_correct = housing_df['rs_prop_yr_built'].map(years_since)
print(yr_built_correct)
housing_df.insert(1, 'rs_prop_yrs_since_built',yr_built_correct)
housing_df = housing_df.drop(columns=['rs_prop_yr_built'])

#first tried fillna(0) but had bad accuracy-- replaced w mean of columns, didn't change much
for column_name in housing_df:
  housing_df[column_name].fillna(housing_df[column_name].mean(),inplace=True)

print(housing_df.head)


#outcome column: violation_count


0         301
1         301
2         236
3         316
4         227
         ... 
315288      9
315289     22
315290     20
315291     16
315292     25
Name: rs_prop_yr_built, Length: 315293, dtype: int64
<bound method NDFrame.head of         rs_prop_sam_id  rs_prop_yrs_since_built  rs_prop_yr_remod  \
0               407453                      301       2010.000000   
1               407452                      301       2010.000000   
2                62318                      236          0.000000   
3               133181                      316          0.000000   
4                73295                      227       2014.000000   
...                ...                      ...               ...   
315288          186426                        9          0.000000   
315289           80028                       22       1523.764783   
315290           27664                       20       1523.764783   
315291          341536                       16       1523.764783   
3152

/tmp/ipykernel_3499/1256943740.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  housing_df[column_name].fillna(housing_df[column_name].mean(),inplace=True)


In [54]:
housing_df = housing_df.drop(columns=['json_bldg_mhl_housing_5year','json_mhl_housing_5year','rs_prop_yr_remod'])


In [55]:
#total split: 64% of data to train each classification and regression, 16% to test each, and 20% to test the two-step model
X = housing_df.drop(columns=['violation_count'])
y = housing_df['violation_count']

from sklearn.model_selection import train_test_split
X_train_total, X_test_2step, y_train_total, y_test_2step = train_test_split(X, y, test_size=0.2, random_state=42)



X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(X_train_total, y_train_total, test_size = 0.2, random_state=42)

#below are the training/testing data to use for classifier since we want to have it train on just 0's and 1's

X_train_cla = X_train_reg.copy()

X_test_cla = X_test_reg.copy()

y_train_cla = y_train_reg.copy()

y_test_cla = y_test_reg.copy()

#TODO drop rows with no violations for regression - get indices from y where violations are not 0 then slice X using those indices
#X_train_reg = X_train_reg[X_train_reg['violat']]


#change classifier data to only 0 or 1
def binary(x):
  if(x==0):
    return 0
  else:
    return 1

print(y_train_cla.head)
y_train_cla = y_train_cla.map(binary)
y_test_cla = y_test_cla.map(binary)

print(y_train_cla.head)


X_train_total.shape, X_test_2step.shape, X_train_one.shape, X_test_one.shape

<bound method NDFrame.head of 253205    0
116626    0
292726    1
51576     4
153916    0
         ..
81245     0
175956    0
141474    2
239801    0
198931    7
Name: violation_count, Length: 201787, dtype: int64>
<bound method NDFrame.head of 253205    0
116626    0
292726    1
51576     1
153916    0
         ..
81245     0
175956    0
141474    1
239801    0
198931    1
Name: violation_count, Length: 201787, dtype: int64>


((252234, 48), (63059, 48), (201787, 51), (50447, 51))

In [57]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression()
log_reg.fit(X_train_cla, y_train_cla)
y_pred_cla = log_reg.predict(X_test_cla)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [58]:
from sklearn import metrics
cnf_matrix = metrics.confusion_matrix(y_test_cla, y_pred_cla)
cnf_matrix

array([[25395,  4666],
       [13562,  6824]])

In [59]:
from sklearn.metrics import classification_report
print(classification_report(y_test_cla, y_pred_cla))

              precision    recall  f1-score   support

           0       0.65      0.84      0.74     30061
           1       0.59      0.33      0.43     20386

    accuracy                           0.64     50447
   macro avg       0.62      0.59      0.58     50447
weighted avg       0.63      0.64      0.61     50447



In [37]:
from sklearn.linear_model import LinearRegression
lin_reg = LinearRegression()
lin_reg.fit(X_train_reg, y_train_reg)